# BDT tutorial — H→ZZ→4μ (Run3 2022 postEE)

End-to-end walk-through of an xgboost binary classifier (signal = ggH→ZZ→4μ, background = qqZZ + DY + TT̅).

Pre-condition: the parquet ntuples in `data/ntuples/*_features.parquet` already exist (skim + feature engineering done by `src/skim.py` and `src/features.py`). This notebook starts from those files.

**Sections**
1. Imports & configuration
2. Load the engineered ntuples
3. Build per-event weights (xsec vs. training)
4. Train / val / test split
5. Train the BDT
6. Training history
7. ROC curve & test AUC
8. Feature importance (gain)
9. Overtraining check (train vs. test score, KS test)
10. Hyperparameter intuition: what `max_depth` does
11. Deploying to SKNanoAnalyzer (xgboost → ONNX → C++ inference)

## 1. Imports & configuration

In [ ]:
import os, sys
from pathlib import Path

# Make the project's `src/` importable so we can reuse FEATURES, etc.
WORKDIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()     # <<< project root
sys.path.append(str(WORKDIR))                       # <<< so `from src.features import FEATURES` works

import numpy as np
import pandas as pd
import awkward as ak
import matplotlib.pyplot as plt

import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import ks_2samp

from src.features import FEATURES                  # <<< canonical training feature list (23 columns)
from src.utils import load_yaml, detect_gpu, SEED  # <<< SEED = 42 used everywhere

print(f"# of features: {len(FEATURES)}")
print(FEATURES)

In [ ]:
# Load the YAML configs so we don't hard-code numbers in the notebook.
samples_cfg   = load_yaml(WORKDIR / "config/samples.yaml")    # <<< xsec, paths, labels per sample
selection_cfg = load_yaml(WORKDIR / "config/selection.yaml")  # <<< BDT hyperparameters and split fractions

LUMI_FB = samples_cfg["lumi_fb"]   # <<< 26.7 fb^-1 for 2022 postEE
BDT_CFG = dict(selection_cfg["bdt"])
TRAIN_CFG = selection_cfg["train"]

# Where to write notebook plots (kept separate from the production `plots/` folder).
OUTDIR = WORKDIR / "notebooks" / "plots"
OUTDIR.mkdir(parents=True, exist_ok=True)

print("Lumi:", LUMI_FB, "fb^-1")
print("BDT cfg:", BDT_CFG)
print("Split cfg:", TRAIN_CFG)

## 2. Load the engineered ntuples

Each parquet file is one MC sample after the 4μ skim with engineered features (m4l, mZ1, mZ2, helicity angles, …) appended. We concatenate them into a single `pandas.DataFrame` and tag each row with its class label (1 = signal, 0 = background) and source sample name (we'll need that for weighting).

In [ ]:
ntuple_dir = WORKDIR / "data/ntuples"

frames = []
for category in ("signal", "background"):                        # <<< both classes live in samples.yaml
    for name, cfg in samples_cfg[category].items():
        fp = ntuple_dir / f"{name}_features.parquet"
        if not fp.exists():
            print(f"  missing {fp.name}, skipping")
            continue
        # awkward is the native format from the skim; convert to a flat DataFrame
        # because every column we care about is one scalar per event.
        arr = ak.from_parquet(fp)
        df  = pd.DataFrame({k: ak.to_numpy(arr[k]) for k in arr.fields})
        df["label"]  = cfg["label"]                              # <<< 1 for ggH, 0 for the rest
        df["sample"] = name                                       # <<< keep the source so we can balance later
        df["xsec"]   = cfg["xsec"]                                # <<< pb, pinned in CLAUDE.md
        frames.append(df)
        print(f"  loaded {name:<12} : {len(df):>7d} events  (xsec = {cfg['xsec']} pb)")

df = pd.concat(frames, ignore_index=True)
print(f"\nTotal: {len(df)} events,  signal: {(df.label==1).sum()},  background: {(df.label==0).sum()}")
df.head()

## 3. Build per-event weights

We need **two** different weights, and confusing them is the most common BDT bug in HEP:

- `xsec_weight` — the physical, lumi-scaled yield weight, with sign preserved (NLO MC samples like DY have negative `genWeight` for some events). Used to make plots look like the dataset and to compute event yields.
- `train_weight` — the weight handed to xgboost. Must be ≥ 0 and **must not** be proportional to xsec, otherwise the BDT spends all its capacity learning the largest-xsec sample (DY here, σ = 6225 pb) and ignores qqZZ which is what actually faces the signal.

The recipe below (per-sample equalisation → per-class equalisation → mean-1 rescale) is the same as `src/train.py::load_dataset`.

In [ ]:
# --- xsec_weight: lumi * xsec * genWeight / sum(genWeight) per sample ----
df["xsec_weight"] = 0.0
df["abs_weight"]  = 0.0
for name in df["sample"].unique():
    sel = df["sample"] == name
    gw  = df.loc[sel, "genWeight"].to_numpy()
    xs  = df.loc[sel, "xsec"].iloc[0]
    sum_gw_signed = float(np.sum(gw))
    sum_gw_abs    = float(np.sum(np.abs(gw)))
    # signed: preserves NLO cancellations → sum(xsec_weight) over the sample = lumi * xsec
    df.loc[sel, "xsec_weight"] = LUMI_FB * 1000.0 * xs * gw / max(abs(sum_gw_signed), 1.0)
    # unsigned: xgboost requires sample_weight >= 0
    df.loc[sel, "abs_weight"]  = LUMI_FB * 1000.0 * xs * np.abs(gw) / max(sum_gw_abs, 1.0)

yield_table = df.groupby("sample").agg(
    n_events=("label", "size"),
    sum_xsec_weight=("xsec_weight", "sum"),  # <<< should equal lumi * xsec ~ N expected events
)
yield_table

In [ ]:
# --- train_weight: each sample contributes equally inside its class --------
df["train_weight"] = 0.0
for label_value in (0, 1):
    cls = df.label == label_value
    names = df.loc[cls, "sample"].unique()
    n_samples_in_class = max(len(names), 1)
    for name in names:
        sel = cls & (df["sample"] == name)
        s = df.loc[sel, "abs_weight"].sum()
        # Each sample's weights sum to 1 / n_samples_in_class → each class sums to 1.
        df.loc[sel, "train_weight"] = df.loc[sel, "abs_weight"] / max(s, 1e-12) / n_samples_in_class

# Rescale so the mean per-event weight is 1. With ~5e5 events the per-event weights
# would otherwise be ~1e-6, which makes hyperparameters like min_child_weight=5
# completely unreachable → the BDT can't grow any tree.
df["train_weight"] *= len(df) / df["train_weight"].sum()

df.groupby(["label", "sample"]).agg(
    n_events=("label", "size"),
    sum_train_weight=("train_weight", "sum"),
    mean_train_weight=("train_weight", "mean"),
)

## 4. Train / val / test split

Two-step stratified split so both classes are represented in every set:
- 75% train+val, 25% test
- of the train+val, 10% goes to validation (for early stopping)

In [ ]:
X = df[FEATURES].to_numpy(dtype=np.float32)         # <<< matrix of inputs, columns in the order of FEATURES
y = df["label"].to_numpy(dtype=np.int64)            # <<< 0 / 1
w = df["train_weight"].to_numpy(dtype=np.float32)   # <<< the BDT weight (NOT xsec_weight)

seed = TRAIN_CFG.get("seed", SEED)

# First peel off the test set (held out, never seen during training).
X_tv, X_test, y_tv, y_test, w_tv, w_test = train_test_split(
    X, y, w,
    test_size=TRAIN_CFG["test_size"], stratify=y, random_state=seed,
)
# Then split train+val into train and val.
X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split(
    X_tv, y_tv, w_tv,
    test_size=TRAIN_CFG["val_size"], stratify=y_tv, random_state=seed,
)

print(f"train: {len(y_tr):>7d}  ({(y_tr==1).sum()} sig)")
print(f"val  : {len(y_val):>7d}  ({(y_val==1).sum()} sig)")
print(f"test : {len(y_test):>7d}  ({(y_test==1).sum()} sig)")

## 5. Train the BDT

xgboost in `tree_method='hist'` mode — fast and the only mode that supports `device='cuda'`. We watch AUC on both the train and val sets every boosting round; early stopping kicks in when val AUC has not improved for 30 rounds.

In [ ]:
use_gpu = TRAIN_CFG.get("use_gpu", True) and detect_gpu()
print("device:", "cuda" if use_gpu else "cpu")

# Pull the early-stopping & metric out of the YAML dict before passing the rest as **kwargs.
bdt_cfg = dict(BDT_CFG)
early_stopping = bdt_cfg.pop("early_stopping_rounds", 30)
eval_metric    = bdt_cfg.pop("eval_metric", "auc")

model = xgb.XGBClassifier(
    **bdt_cfg,                              # <<< n_estimators, max_depth, learning_rate, ...
    objective="binary:logistic",            # <<< binary classification with sigmoid output
    eval_metric=eval_metric,                # <<< 'auc' — monitor area under ROC each round
    early_stopping_rounds=early_stopping,   # <<< stop if val AUC stalls for 30 rounds
    device="cuda" if use_gpu else "cpu",
    tree_method="hist",
    random_state=seed,
)

model.fit(
    X_tr, y_tr,
    sample_weight=w_tr,                                         # <<< class-balanced training weight
    eval_set=[(X_tr, y_tr), (X_val, y_val)],                    # <<< validation_0 = train, validation_1 = val
    sample_weight_eval_set=[w_tr, w_val],                       # <<< weighted AUC on both sets
    verbose=False,
)

best_iter = getattr(model, "best_iteration", None)
print(f"best iter = {best_iter} / {bdt_cfg['n_estimators']}")

## 6. Training history

Train and val AUC vs. boosting round. If val flattens (or starts dropping) while train keeps climbing, that's the classic overfitting fingerprint and early stopping will pick the iteration before the divergence.

In [ ]:
hist = model.evals_result()
train_curve = hist["validation_0"][eval_metric]
val_curve   = hist["validation_1"][eval_metric]
iters = np.arange(1, len(train_curve) + 1)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(iters, train_curve, label="train")
ax.plot(iters, val_curve,   label="val")
if best_iter is not None:
    ax.axvline(best_iter + 1, color="grey", ls="--", lw=1, label=f"best iter ({best_iter+1})")
ax.set_xlabel("boosting round")
ax.set_ylabel(eval_metric)
ax.set_title("BDT training history")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUTDIR / "training_history.png", dpi=200)
plt.show()

## 7. ROC curve & test AUC

Two AUCs reported on the held-out test set:
- **weighted** — with `train_weight` (matches the training metric, treats all samples equally).
- **unweighted** — plain event count, dominated by whichever sample has the most surviving events.

They usually differ by a few %; the gap tells you how much the per-sample balancing is doing.

In [ ]:
p_test = model.predict_proba(X_test)[:, 1]   # <<< column 1 = P(signal)

fpr_w, tpr_w, _ = roc_curve(y_test, p_test, sample_weight=w_test)
auc_w = roc_auc_score(y_test, p_test, sample_weight=w_test)
fpr_u, tpr_u, _ = roc_curve(y_test, p_test)
auc_u = roc_auc_score(y_test, p_test)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(fpr_w, tpr_w, label=f"weighted   AUC = {auc_w:.4f}")
ax.plot(fpr_u, tpr_u, label=f"unweighted AUC = {auc_u:.4f}", linestyle="--")
ax.plot([0, 1], [0, 1], color="grey", linestyle=":", lw=1)   # <<< random-guess baseline
ax.set_xlabel("background efficiency (FPR)")
ax.set_ylabel("signal efficiency (TPR)")
ax.set_title("ROC — held-out test set")
ax.legend(loc="lower right"); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUTDIR / "roc.png", dpi=200)
plt.show()
print(f"AUC weighted   = {auc_w:.4f}")
print(f"AUC unweighted = {auc_u:.4f}")

## 8. Feature importance (gain)

xgboost reports several importance types. **Gain** is the most physically meaningful: the average improvement in the loss whenever a feature is used to split a node. Higher = the BDT relies on it more.

Don't be surprised if `m4l` dominates — the Higgs mass peak is the biggest discriminant available. (See `CLAUDE.md → Features`: we leave it in for this learning project but it would be removed for a mass-window analysis to avoid trivially learning the peak.)

In [ ]:
booster = model.get_booster()
raw = booster.get_score(importance_type="gain")             # <<< keys are 'f0','f1',... — raw column index
importances = {FEATURES[int(k[1:])]: v for k, v in raw.items()}
series = pd.Series(importances).reindex(FEATURES, fill_value=0.0).sort_values()

fig, ax = plt.subplots(figsize=(7, 0.3 * len(FEATURES) + 1.5))
ax.barh(series.index, series.values)
ax.set_xlabel("gain")
ax.set_title("BDT feature importance (gain)")
ax.grid(alpha=0.3, axis="x")
fig.tight_layout()
fig.savefig(OUTDIR / "feature_importance.png", dpi=200)
plt.show()
series.sort_values(ascending=False).head(8)

## 9. Overtraining check

Compare the BDT score distribution on **train** vs. **test**, separately for signal and background. If the model is overtrained, train and test will disagree (especially in the tails).

We also run a Kolmogorov–Smirnov test: a high p-value (≫ 0.05) means we cannot reject the hypothesis that train and test were drawn from the same distribution — i.e. no overtraining detected. Note: `ks_2samp` is unweighted (scipy doesn't support weights here), so it's a diagnostic, not a precise statement.

In [ ]:
p_train = model.predict_proba(X_tr)[:, 1]
p_test  = model.predict_proba(X_test)[:, 1]   # already computed above, recompute for clarity

bins = np.linspace(0.0, 1.0, 41)
splits = {
    "sig train": (p_train[y_tr == 1], w_tr[y_tr == 1]),
    "sig test" : (p_test [y_test == 1], w_test[y_test == 1]),
    "bkg train": (p_train[y_tr == 0], w_tr[y_tr == 0]),
    "bkg test" : (p_test [y_test == 0], w_test[y_test == 0]),
}
styles = {
    "sig train": dict(color="C0", linestyle="-",  histtype="step", lw=1.5),
    "sig test" : dict(color="C0", linestyle="--", histtype="step", lw=1.5),
    "bkg train": dict(color="C3", linestyle="-",  histtype="step", lw=1.5),
    "bkg test" : dict(color="C3", linestyle="--", histtype="step", lw=1.5),
}

fig, ax = plt.subplots(figsize=(7, 5))
for label, (vals, wts) in splits.items():
    ax.hist(vals, bins=bins, weights=wts, density=True, label=label, **styles[label])

ks_sig = ks_2samp(splits["sig train"][0], splits["sig test"][0])
ks_bkg = ks_2samp(splits["bkg train"][0], splits["bkg test"][0])

ax.set_xlabel("BDT score")
ax.set_ylabel("normalized density")
ax.set_yscale("log")
ax.set_title(f"Overtraining — KS sig p={ks_sig.pvalue:.3g}, bkg p={ks_bkg.pvalue:.3g}")
ax.legend(loc="upper center"); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUTDIR / "overtraining.png", dpi=200)
plt.show()
print(f"KS signal     stat={ks_sig.statistic:.3g}  p={ks_sig.pvalue:.3g}")
print(f"KS background stat={ks_bkg.statistic:.3g}  p={ks_bkg.pvalue:.3g}")

## 10. Hyperparameter intuition: what `max_depth` does

Same dataset, same everything — just sweep `max_depth`. Shallower trees underfit (low train AND low val AUC); deeper trees can overfit (train climbs faster than val). For this particular dataset early stopping makes a deep tree mostly safe, but the shape of the curves is the lesson.

Run with a small `n_estimators` to keep the sweep fast.

In [ ]:
depths = [2, 3, 5, 8]
results = []
for d in depths:
    m = xgb.XGBClassifier(
        n_estimators=200, max_depth=d, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
        objective="binary:logistic", eval_metric="auc",
        early_stopping_rounds=20,
        device="cuda" if use_gpu else "cpu", tree_method="hist",
        random_state=seed,
    )
    m.fit(X_tr, y_tr, sample_weight=w_tr,
          eval_set=[(X_tr, y_tr), (X_val, y_val)],
          sample_weight_eval_set=[w_tr, w_val],
          verbose=False)
    p_tr  = m.predict_proba(X_tr)[:, 1]
    p_val = m.predict_proba(X_val)[:, 1]
    p_te  = m.predict_proba(X_test)[:, 1]
    results.append({
        "max_depth": d,
        "best_iter": m.best_iteration,
        "AUC train": roc_auc_score(y_tr,   p_tr,  sample_weight=w_tr),
        "AUC val"  : roc_auc_score(y_val,  p_val, sample_weight=w_val),
        "AUC test" : roc_auc_score(y_test, p_te,  sample_weight=w_test),
    })
pd.DataFrame(results).set_index("max_depth")

---
## 11. Deploying to SKNanoAnalyzer

The BDT we just trained needs to run inside SKNanoAnalyzer's C++ analyzer loop. SKNanoAnalyzer already ships an inference helper, `AnalyzerTools/{include,src}/MLHelper.{h,cc}`, that wraps `onnxruntime-cpp` — but it loads **ONNX**, not xgboost's native JSON. So the deployment chain is:

```
data/models/bdt_v1.json   (xgboost native, this notebook)
        ↓  onnxmltools.convert_xgboost
data/models/bdt_v1.onnx   (portable, what MLHelper loads)
        ↓  copy into SKNanoAnalyzer/data/<era>/HZZ4mu/
        ↓  new analyzer calls MLHelper::Run_ONNX_Model(...)
BDT score per event in the analyzer's output tree
```

Sub-sections:
- 11.1 Install the converters
- 11.2 Export `bdt_v1.json` → `bdt_v1.onnx`
- 11.3 Numerical parity check (xgboost vs. ONNX) — **do not skip**
- 11.4 Drop the file into SKNanoAnalyzer + the one-line CMake change
- 11.5 The analyzer skeleton (`HZZ4muBDT.{h,cc}`)
- 11.6 C++/Python parity in production

### 11.1 Install the converters

`onnxmltools` does the xgboost → ONNX conversion; `onnx` and `onnxruntime` are needed to validate the output. Add these to `requirements.txt`.

In [ ]:
# Run once per environment.
%pip install --quiet onnxmltools onnxconverter-common onnx onnxruntime

### 11.2 Export `bdt_v1.json` → `bdt_v1.onnx`

Two settings are non-negotiable:

- **One named input tensor.** Pick a name (`features` here) and remember it — the C++ side looks up inputs by name in `MLHelper::Run_ONNX_Model`.
- **Disable ZipMap.** Without `zipmap=False`, xgboost's ONNX export emits the per-class probabilities as a `Map<int64,float>`. `MLHelper` only handles `FloatArray` outputs and will throw at run time. With `zipmap=False` you get a plain `(N, 2)` `float32` tensor whose column 1 is the signal probability.

In [ ]:
from pathlib import Path
import xgboost as xgb
from onnxmltools.convert import convert_xgboost
from onnxconverter_common.data_types import FloatTensorType
from src.features import FEATURES

WORKDIR  = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
JSON_PATH = WORKDIR / "data/models/bdt_v1.json"
ONNX_PATH = WORKDIR / "data/models/bdt_v1.onnx"
FEAT_TXT  = WORKDIR / "data/models/bdt_v1_features.txt"

model = xgb.XGBClassifier()
model.load_model(JSON_PATH)

# Single named float input — shape [batch, n_features].
initial_type = [("features", FloatTensorType([None, len(FEATURES)]))]
onnx_model = convert_xgboost(
    model,
    initial_types=initial_type,
    target_opset=15,           # <<< stay <= the onnxruntime in the SKNano env
    options={id(model): {"zipmap": False}},  # <<< plain (N,2) tensor output
)
ONNX_PATH.write_bytes(onnx_model.SerializeToString())

# Pin the feature order alongside the model. The C++ side MUST fill the
# input tensor in this exact order or the BDT score is meaningless.
FEAT_TXT.write_text("\n".join(FEATURES))

print("wrote", ONNX_PATH, "(", ONNX_PATH.stat().st_size, "bytes )")
print("wrote", FEAT_TXT)

### 11.3 Numerical parity check

Always validate that ONNX agrees with xgboost on the test set before shipping. A 1e-5 max-difference is the realistic bar; anything larger usually means an opset mismatch or the wrong float dtype on the inputs.

Also dump ~100 representative events to a CSV — this is the ground truth the C++ analyzer reproduces in section 11.6.

In [ ]:
import numpy as np
import onnxruntime as ort

sess = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
print("ONNX inputs :", [i.name for i in sess.get_inputs()])
print("ONNX outputs:", [o.name for o in sess.get_outputs()])  # <<< note these names — C++ uses them

# `X_test` is already in float32 from earlier in the notebook.
p_xgb  = model.predict_proba(X_test)[:, 1]
outs   = sess.run(None, {"features": X_test})
# With zipmap=False the probability tensor is outs[1] of shape (N, 2).
p_onnx = outs[1][:, 1]

max_abs = float(np.max(np.abs(p_xgb - p_onnx)))
print(f"max |Δ probability| = {max_abs:.2e}")
assert max_abs < 1e-5, "ONNX export disagrees with xgboost — do NOT ship this file"

In [ ]:
# Dump 100 events as a C++ regression baseline. Columns:
#   <FEATURES>, py_score, onnx_score
# The SKNanoAnalyzer analyzer reads its features from NanoAOD, builds the
# same vector, and must match `onnx_score` to ~1e-6.
import pandas as pd
rng = np.random.default_rng(seed)
idx = rng.choice(len(X_test), size=100, replace=False)
VAL_CSV = WORKDIR / "data/models/bdt_v1_validation.csv"
val = pd.DataFrame(X_test[idx], columns=FEATURES)
val["py_score"]   = p_xgb[idx]
val["onnx_score"] = p_onnx[idx]
val.to_csv(VAL_CSV, index=False)
print("wrote", VAL_CSV)
val.head()

### 11.4 Drop the file into SKNanoAnalyzer

Copy the validated `.onnx` into a versioned subdirectory inside SKNanoAnalyzer's `data/<era>/`:

```bash
SKNANO=/path/to/your/SKNanoAnalyzer
DEST=$SKNANO/data/Run3_v15_Run2_v15/2022EE/HZZ4mu
mkdir -p $DEST
cp data/models/bdt_v1.onnx              $DEST/
cp data/models/bdt_v1_features.txt       $DEST/
cp data/models/bdt_v1_validation.csv     $DEST/
```

`MLHelper` is built but **not yet linked into the Analyzers library** — currently `Analyzers/CMakeLists.txt:44` reads `#target_link_libraries(Analyzers PUBLIC MLHelper)`. Uncomment that line:

```cmake
target_link_libraries(Analyzers PUBLIC MLHelper)
```

Then `./scripts/rebuild.sh`. (You're the first user of `MLHelper` from `Analyzers/`; without this change the linker will not find the symbols.)

### 11.5 The analyzer skeleton

Following SKNanoAnalyzer's `CLAUDE.md` pattern: header in `Analyzers/include/`, source in `Analyzers/src/`, register in `Analyzers/include/AnalyzersLinkDef.hpp`, inherit from `AnalyzerCore`.

**`Analyzers/include/HZZ4muBDT.h`**
```cpp
#ifndef HZZ4MUBDT_H
#define HZZ4MUBDT_H
#include "AnalyzerCore.h"
#include "MLHelper.h"
#include <memory>

class HZZ4muBDT : public AnalyzerCore {
public:
    void initializeAnalyzer() override;
    void executeEvent() override;
    ~HZZ4muBDT() override = default;
private:
    std::unique_ptr<MLHelper> bdt_;
    static constexpr int N_FEAT = 23;   // == len(FEATURES) in src/features.py
};
#endif
```

**`Analyzers/src/HZZ4muBDT.cc`** (illustrative — the selection helpers must mirror `config/selection.yaml` exactly):
```cpp
#include "HZZ4muBDT.h"

void HZZ4muBDT::initializeAnalyzer() {
    TString p = TString(getenv("SKNANO_DATA"))
             + "/2022EE/HZZ4mu/bdt_v1.onnx";
    bdt_ = std::make_unique<MLHelper>(p.Data(), MLHelper::ModelType::ONNX);
}

void HZZ4muBDT::executeEvent() {
    // 1. Trigger OR — same list as config/selection.yaml::triggers.
    // 2. RVec<Muon> mus = GetMuons("loose", 5.0, 2.4);
    //    Apply iso<0.35, dxy/dz/sip3d, charge sum 0, exactly 4.
    // 3. Z1/Z2 pairing: closest-to-mZ for Z1, mZ1 ∈ [40,120], mZ2 ∈ [12,120].
    // 4. m4l ∈ [70, 1000].
    // 5. Build the 23-feature vector in the SAME order as FEATURES.

    FloatArray feats(N_FEAT);
    feats[0] = m4l;   feats[1] = mZ1;   feats[2] = mZ2;
    feats[3] = pt4l;  feats[4] = eta4l;
    feats[5] = ptZ1;  feats[6] = ptZ2;  feats[7] = dR_Z1Z2;
    /* ... per-muon pt/eta, ratios, helicity angles ... */

    std::unordered_map<std::string, VariousArray> in {
        {"features", feats}                                    // <<< name from 11.2
    };
    std::unordered_map<std::string, IntArray> in_shape {
        {"features", IntArray{1, N_FEAT}}
    };
    auto out = bdt_->Run_ONNX_Model(in, in_shape);

    // Output node name comes from sess.get_outputs() in 11.3 —
    // for xgboost+zipmap=False it is typically "probabilities".
    float p_sig = out["probabilities"][1];                     // class 1 = signal
    // → fill histograms / cut / write to output tree
}
```

Then add to `Analyzers/include/AnalyzersLinkDef.hpp`:
```cpp
#pragma link C++ class HZZ4muBDT+;
```
Rebuild and run:
```bash
./scripts/rebuild.sh
SKNano.py -a HZZ4muBDT -i 'GluGluHto2Zto4L*' -e 2022EE -n 10
```

### 11.6 C++/Python parity in production

This is where deployment usually goes wrong. Take the events in `bdt_v1_validation.csv`, run them through the SKNanoAnalyzer analyzer, and assert column-by-column:

- each of the 23 features matches the Python value to ~1e-5 (float32 round-trip);
- the BDT score matches `onnx_score` to ~1e-6 (same ONNX runtime on both sides).

If features match but the score doesn't, suspect the input/output tensor name or `zipmap`. If features themselves don't match, the most common culprits are:

- **pT-sort tie-breaking** on the muon collection (NanoAOD already orders by pT, but enforce it explicitly in the analyzer).
- **Z₁/Z₂ pairing ambiguity** — the rule is *Z₁ = pair closest to mZ*; for the tie case, document which pairing wins.
- **ℓ⁻ choice in each Z** — the helicity angles `Φ`, `Φ₁` flip sign if you pick the wrong lepton. See `src/features.py::add_helicity_angles` (the negative-charge muon is the reference in each Z).
- **Beam-axis sign convention** in `cosθ*` (we use +z in the lab frame).
- **Float precision** in the boost chain — do all 4-vector arithmetic in `double` and cast to `float` only when filling the input tensor.

A sign flip in one helicity angle will silently shift the BDT score without raising any error — the parity CSV is the only thing that catches it.

## What to try next
- Section **11** below walks through ONNX export and SKNanoAnalyzer integration when you're ready to ship.
